# YOLO Fire Detector - Persistent Cloud Training

Notebook operativo per eseguire in cloud l'intera pipeline del progetto: sync del repository, generazione o riuso del dataset, training YOLOv8, registrazione dell'export finale e creazione dello zip da riportare in locale.

## Prima di iniziare

1. Da VS Code locale prepara il bundle con `python prepare_cloud_bundle.py`.
2. Verifica che esista `dist/yolo-fire-detector-cloud.zip`.
3. Apri questo notebook in Colab.
4. Se vuoi usare la GPU, abilitala in `Runtime -> Change runtime type`.
5. Esegui le celle in ordine, senza lanciare separatamente `generator.py` o `train.py`.

## Cosa fa questo notebook

Questo notebook usa una root persistente unica e mantiene nello stesso punto:

- codice del progetto
- dataset generati e relativi manifest YAML
- run di training e diagnostica YOLO
- export finali e registry degli export

Su Colab monta Google Drive e riparte automaticamente dallo stesso stato se esiste gia' un dataset compatibile o una run riprendibile.

## Root persistente e bundle

Il notebook cerca automaticamente `yolo-fire-detector-cloud.zip` e lo puo' trovare in uno di questi posti:

- `/content`
- Google Drive

Se trova il file fuori dalla root persistente, lo sposta in `inputs/` e aggiorna la copia del repository in `repo/`.

In [1]:
import json

import os

from pathlib import Path

import shutil

import sys

import zipfile


IN_COLAB = 'google.colab' in sys.modules

BUNDLE_NAME = 'yolo-fire-detector-cloud.zip'

if IN_COLAB:

    from google.colab import drive

    drive.mount('/content/drive')

    PERSISTENT_ROOT = Path('/content/drive/MyDrive/yolo-fire-detector')

else:

    PERSISTENT_ROOT = Path('/content/yolo-fire-detector')


REPO_ROOT = PERSISTENT_ROOT / 'repo'

INPUTS_ROOT = PERSISTENT_ROOT / 'inputs'

RUNTIME_CONFIG_PATH = REPO_ROOT / 'configs' / 'cloud.runtime.yaml'


PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

INPUTS_ROOT.mkdir(parents=True, exist_ok=True)


print('IN_COLAB =', IN_COLAB)

print('BUNDLE_NAME =', BUNDLE_NAME)

print('PERSISTENT_ROOT =', PERSISTENT_ROOT)

print('REPO_ROOT =', REPO_ROOT)

print('INPUTS_ROOT =', INPUTS_ROOT)

print('RUNTIME_CONFIG_PATH =', RUNTIME_CONFIG_PATH)

IN_COLAB = False
BUNDLE_NAME = yolo-fire-detector-cloud.zip
PERSISTENT_ROOT = \content\yolo-fire-detector
REPO_ROOT = \content\yolo-fire-detector\repo
INPUTS_ROOT = \content\yolo-fire-detector\inputs
RUNTIME_CONFIG_PATH = \content\yolo-fire-detector\repo\configs\cloud.runtime.yaml


## Sync progetto nella root persistente

Questa sezione prepara il contesto di lavoro usato da tutte le celle successive.

La cella qui sotto fa automaticamente questo:

1. controlla se il bundle `yolo-fire-detector-cloud.zip` e' gia' presente in `inputs/`
2. se non c'e', lo cerca nel contesto locale della sessione e in Google Drive
3. se lo trova fuori dalla root persistente, lo sposta in `<PERSISTENT_ROOT>/inputs/`
4. estrae o aggiorna il repository in `<PERSISTENT_ROOT>/repo/`

Percorsi principali usati dal notebook:

- `PERSISTENT_ROOT`: root persistente del lavoro
- `INPUTS_ROOT`: cartella dove viene conservato lo zip del progetto
- `REPO_ROOT`: cartella del repository estratto
- `RUNTIME_CONFIG_PATH`: file `configs/cloud.runtime.yaml` generato dal notebook

### Quando intervenire manualmente

- Se vuoi forzare l'aggiornamento del codice estratto, imposta `FORCE_PROJECT_REFRESH = True`.
- Se il notebook non trova lo zip, metti `yolo-fire-detector-cloud.zip` in `/content` o in Google Drive e riesegui questa sezione.
- Non serve estrarre il bundle a mano.

In [4]:
FORCE_PROJECT_REFRESH = False


def find_bundle_candidates(bundle_name: str) -> list[Path]:
    candidates: list[Path] = []

    def add_candidate(path: Path) -> None:
        if path.exists() and path not in candidates:
            candidates.append(path)

    add_candidate(INPUTS_ROOT / bundle_name)

    local_search_roots = [Path.cwd(), Path('/content')]
    for root in local_search_roots:
        if not root.exists():
            continue
        add_candidate(root / bundle_name)

    if IN_COLAB:
        drive_root = Path('/content/drive/MyDrive')
        if drive_root.exists():
            add_candidate(drive_root / bundle_name)
            for match in drive_root.rglob(bundle_name):
                add_candidate(match)

    return candidates


def ensure_bundle_in_persistent_inputs(bundle_name: str) -> Path | None:
    candidates = find_bundle_candidates(bundle_name)
    if not candidates:
        return None

    chosen_bundle = candidates[0]
    persistent_bundle = INPUTS_ROOT / bundle_name

    if chosen_bundle.resolve() != persistent_bundle.resolve():
        INPUTS_ROOT.mkdir(parents=True, exist_ok=True)
        if persistent_bundle.exists():
            persistent_bundle.unlink()
        shutil.move(str(chosen_bundle), str(persistent_bundle))
        print('Bundle spostato in root persistente:', persistent_bundle)
    else:
        print('Bundle gia presente in root persistente:', persistent_bundle)

    return persistent_bundle


bundle_path = ensure_bundle_in_persistent_inputs(BUNDLE_NAME)
repo_ready = (REPO_ROOT / 'run_experiment.py').exists()


if bundle_path is None and not repo_ready:
    raise FileNotFoundError(
        'Bundle non trovato. Metti yolo-fire-detector-cloud.zip in /content o in Google Drive e rilancia la cella.'
    )


if bundle_path is not None and (FORCE_PROJECT_REFRESH or not repo_ready):
    temp_extract_dir = PERSISTENT_ROOT / '_incoming_repo'

    shutil.rmtree(temp_extract_dir, ignore_errors=True)
    shutil.rmtree(REPO_ROOT, ignore_errors=True)

    temp_extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(bundle_path, 'r') as archive:
        archive.extractall(temp_extract_dir)

    extracted_files = [child for child in temp_extract_dir.iterdir() if child.is_file()]
    extracted_roots = [child for child in temp_extract_dir.iterdir() if child.is_dir()]
    if extracted_files:
        source_root = temp_extract_dir
    elif len(extracted_roots) == 1:
        source_root = extracted_roots[0]
    else:
        source_root = temp_extract_dir

    shutil.copytree(source_root, REPO_ROOT, dirs_exist_ok=True)
    shutil.rmtree(temp_extract_dir, ignore_errors=True)


print('Bundle persistente =', bundle_path)
print('Repo pronta =', REPO_ROOT.exists())
print('run_experiment.py presente =', (REPO_ROOT / 'run_experiment.py').exists())

Bundle gia presente in root persistente: \content\yolo-fire-detector\inputs\yolo-fire-detector-cloud.zip
Bundle persistente = \content\yolo-fire-detector\inputs\yolo-fire-detector-cloud.zip
Repo pronta = True
run_experiment.py presente = True


## Installazione dipendenze e ambiente di esecuzione

Dopo che il repository e' stato sincronizzato, la cella successiva:

1. entra nella cartella `repo/` estratta dal bundle
2. installa le dipendenze da `requirements.txt`
3. prova a mostrare lo stato GPU con `nvidia-smi`

Questa e' la preparazione standard del runtime cloud. In genere non devi modificare nulla qui.

Se la sessione e' gia' pronta e le dipendenze sono gia' installate, puoi comunque rieseguire la cella senza cambiare il flusso del notebook.

In [5]:
import subprocess



os.chdir(REPO_ROOT)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

subprocess.run(['nvidia-smi'], check=False)

print('Dipendenze installate e working directory impostata su', REPO_ROOT)


Dipendenze installate e working directory impostata su \content\yolo-fire-detector\repo


## Configurazione runtime

Questa e' la parte in cui scegli come costruire `repo/configs/cloud.runtime.yaml`. Hai due modalita'.

### Modalita' A: usa una config gia' pronta

Imposta:

- `USE_READY_CONFIG_AS_IS = True`
- `READY_CONFIG_NAME = 'cloud.recall.yaml'`

In questo caso il notebook usa direttamente quel preset e cambia solo `project.persistent_root`. E' il modo piu' semplice per lanciare un esperimento pronto.

### Modalita' B: parti da un preset e sovrascrivi i campi principali

Imposta:

- `USE_READY_CONFIG_AS_IS = False`
- `BASE_CONFIG_NAME = 'cloud.default.yaml'`

Poi puoi modificare i parametri principali:

- `BASE_CONFIG_NAME`
- `DATASET_LABEL`
- `TRAINING_LABEL`
- `NUM_IMAGES`
- `IMAGE_SIZE`
- `EPOCHS`
- `BATCH_SIZE`
- `MODEL_SIZE`
- `DEVICE`

Alla fine di questa sezione il notebook scrive `cloud.runtime.yaml`, che e' la config effettivamente usata dalla pipeline.

### Note pratiche

- Se vuoi rigenerare il dataset, imposta `dataset.force_regenerate: true` nella config runtime generata o nel preset di partenza.
- Se vuoi evitare il resume, usa `training.resume: never`.
- Se vuoi semplicemente lanciare un preset senza modifiche manuali, resta in Modalita' A.

In [6]:
import yaml


AVAILABLE_BASE_CONFIGS = [
    'cloud.default.yaml',
    'cloud.quick-screen.yaml',
    'cloud.recall.yaml',
    'cloud.robust.yaml',
    'cloud.small-fire.yaml',
    'cloud.hard-negatives.yaml',
    'cloud.capacity.yaml',
]


USE_READY_CONFIG_AS_IS = True
READY_CONFIG_NAME = 'cloud.recall.yaml'


BASE_CONFIG_NAME = 'cloud.default.yaml'
DATASET_LABEL = 'synthetic-fire-cloud'
TRAINING_LABEL = 'yolov8n-cloud'
NUM_IMAGES = 2000
IMAGE_SIZE = 640
EPOCHS = 50
BATCH_SIZE = 16
MODEL_SIZE = 'n'
DEVICE = '0'


selected_config_name = READY_CONFIG_NAME if USE_READY_CONFIG_AS_IS else BASE_CONFIG_NAME
base_config_path = REPO_ROOT / 'configs' / selected_config_name

if selected_config_name not in AVAILABLE_BASE_CONFIGS:
    raise ValueError(f'Preset non supportato: {selected_config_name}')

if not base_config_path.exists():
    raise FileNotFoundError(f'Config base non trovata: {base_config_path}')

with open(base_config_path, 'r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)


config['project']['persistent_root'] = str(PERSISTENT_ROOT)


if not USE_READY_CONFIG_AS_IS:
    config['dataset']['label'] = DATASET_LABEL
    config['dataset']['num_images'] = NUM_IMAGES
    config['dataset']['image_size'] = IMAGE_SIZE
    config['dataset']['force_regenerate'] = False

    config['training']['label'] = TRAINING_LABEL
    config['training']['model_size'] = MODEL_SIZE
    config['training']['epochs'] = EPOCHS
    config['training']['batch_size'] = BATCH_SIZE
    config['training']['image_size'] = IMAGE_SIZE
    config['training']['device'] = DEVICE
    config['training']['resume'] = 'auto'


RUNTIME_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(RUNTIME_CONFIG_PATH, 'w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False, allow_unicode=False)


print('USE_READY_CONFIG_AS_IS =', USE_READY_CONFIG_AS_IS)
print('Selected config =', selected_config_name)
print(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))

USE_READY_CONFIG_AS_IS = True
Selected config = cloud.recall.yaml
project:
  environment: cloud
  label: recall-focus
  persistent_root: \content\yolo-fire-detector
dataset:
  label: synthetic-fire-recall
  fire_image_paths:
  - base_fire_images/fire.png
  num_images: 4000
  image_size: 640
  negative_ratio: 0.2
  train_split: 0.8
  seed: 52
  force_regenerate: false
training:
  label: yolov8s-recall
  model_size: s
  weights: yolov8s.pt
  device: '0'
  epochs: 80
  batch_size: 16
  image_size: 640
  resume: auto
dataset_settings_overrides:
  fire_scale_min: 0.03
  fire_scale_max: 0.6
image_transform_overrides:
  rotation_deg_min: -120
  rotation_deg_max: 120
  perspective_shift: 140
  enable_color_shift: true
  color_shift_prob: 0.5
  augment_negative_backgrounds: true
training_overrides:
  patience: 20
  learning_rate_init: 0.005
  learning_rate_final: 0.01
  momentum: 0.937
  weight_decay: 0.0005
  rotation_degrees: 10
  translate: 0.08
  scale: 0.4
  flip_vertical: 0.5
  flip_horiz

In [7]:
# Optional local smoke override for notebook validation
if not IN_COLAB:
    with open(RUNTIME_CONFIG_PATH, 'r', encoding='utf-8') as handle:
        smoke_config = yaml.safe_load(handle)

    smoke_config['project']['label'] = 'cloud-notebook-smoke'
    smoke_config['dataset']['label'] = 'cloud-notebook-smoke-dataset'
    smoke_config['dataset']['num_images'] = 12
    smoke_config['dataset']['image_size'] = 320
    smoke_config['dataset']['force_regenerate'] = True
    smoke_config['training']['label'] = 'cloud-notebook-smoke'
    smoke_config['training']['model_size'] = 'n'
    smoke_config['training']['weights'] = 'yolov8n.pt'
    smoke_config['training']['epochs'] = 1
    smoke_config['training']['batch_size'] = 4
    smoke_config['training']['image_size'] = 320
    smoke_config['training']['device'] = 'cpu'
    smoke_config['training']['resume'] = 'never'

    with open(RUNTIME_CONFIG_PATH, 'w', encoding='utf-8') as handle:
        yaml.safe_dump(smoke_config, handle, sort_keys=False, allow_unicode=False)

    print('Local notebook smoke override applied:')
    print(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))

Local notebook smoke override applied:
project:
  environment: cloud
  label: cloud-notebook-smoke
  persistent_root: \content\yolo-fire-detector
dataset:
  label: cloud-notebook-smoke-dataset
  fire_image_paths:
  - base_fire_images/fire.png
  num_images: 12
  image_size: 320
  negative_ratio: 0.2
  train_split: 0.8
  seed: 52
  force_regenerate: true
training:
  label: cloud-notebook-smoke
  model_size: n
  weights: yolov8n.pt
  device: cpu
  epochs: 1
  batch_size: 4
  image_size: 320
  resume: never
dataset_settings_overrides:
  fire_scale_min: 0.03
  fire_scale_max: 0.6
image_transform_overrides:
  rotation_deg_min: -120
  rotation_deg_max: 120
  perspective_shift: 140
  enable_color_shift: true
  color_shift_prob: 0.5
  augment_negative_backgrounds: true
training_overrides:
  patience: 20
  learning_rate_init: 0.005
  learning_rate_final: 0.01
  momentum: 0.937
  weight_decay: 0.0005
  rotation_degrees: 10
  translate: 0.08
  scale: 0.4
  flip_vertical: 0.5
  flip_horizontal: 0.5

## Esecuzione della pipeline

La cella seguente esegue l'entrypoint ufficiale del progetto:

```bash
python run_experiment.py --config configs/cloud.runtime.yaml
```

Questo comando orchestra tutto il flusso:

- riuso o generazione del dataset sintetico
- training YOLOv8
- registrazione dell'export finale
- aggiornamento del registry in `exports/latest.yaml`

Non serve eseguire moduli interni come `generator.py` o `train.py` separatamente.

In [8]:
command = [sys.executable, 'run_experiment.py', '--config', str(RUNTIME_CONFIG_PATH)]

print('Executing:', ' '.join(command))

subprocess.run(command, check=True, cwd=REPO_ROOT)


Executing: c:\Users\Windows\Dropbox\__SCUOLA\yolo-fire-detector\.venv\Scripts\python.exe run_experiment.py --config \content\yolo-fire-detector\repo\configs\cloud.runtime.yaml


CompletedProcess(args=['c:\\Users\\Windows\\Dropbox\\__SCUOLA\\yolo-fire-detector\\.venv\\Scripts\\python.exe', 'run_experiment.py', '--config', '\\content\\yolo-fire-detector\\repo\\configs\\cloud.runtime.yaml'], returncode=0)

## Output persistenti e download su PC

Dopo l'esecuzione della pipeline, tutti i risultati restano sotto la stessa root persistente.

Di default, in Colab, la root e':

```text
/content/drive/MyDrive/yolo-fire-detector
```

### `datasets/`

Ogni dataset sta in:

```text
datasets/<dataset-label>-<fingerprint>/
```

Dentro trovi:

- `yolo_dataset.yaml`
- `dataset_manifest.yaml`
- `images/`
- `labels/`

### `runs/`

Ogni run sta in:

```text
runs/<run_label>/
```

Dentro trovi almeno:

- `resolved_config.yaml`
- `pipeline_summary.yaml`
- `training_run.yaml`
- metriche e diagnostica YOLO

Durante il training possono comparire checkpoint in `weights/`. Se la run termina con successo, i checkpoint di resume vengono rimossi automaticamente.

### `exports/`

Qui trovi gli artefatti finali per inferenza:

- `exports/<run_label>.pt`
- `exports/<run_label>.yaml`
- `exports/latest.yaml`

`latest.yaml` punta all'export corrente.

### Download sul PC locale

L'ultima cella del notebook legge `exports/latest.yaml`, comprime tutta la cartella `exports/` in `downloads/yolo-fire-detector-exports.zip` e, in Colab, stampa anche il comando per scaricare lo zip sul PC locale.

### Se riapri una sessione

Riesegui semplicemente il notebook:

- il dataset viene riusato se il fingerprint coincide
- la run puo' riprendere se esiste un checkpoint compatibile e `training.resume` lo consente

In [9]:
latest_registry = PERSISTENT_ROOT / 'exports' / 'latest.yaml'
exports_root = PERSISTENT_ROOT / 'exports'
download_root = PERSISTENT_ROOT / 'downloads'
download_root.mkdir(parents=True, exist_ok=True)
exports_archive = download_root / 'yolo-fire-detector-exports.zip'

if latest_registry.exists():

    print('Latest export registry:')
    print(latest_registry.read_text(encoding='utf-8'))

else:

    print('Nessun export registrato ancora.')


print('\nExports disponibili:')

for path in sorted(exports_root.glob('*')):

    print('-', path)


if exports_root.exists():
    if exports_archive.exists():
        exports_archive.unlink()
    shutil.make_archive(str(exports_archive.with_suffix('')), 'zip', root_dir=exports_root, base_dir='.')
    print('\nArchivio export pronto:', exports_archive)

    if IN_COLAB:
        print('\nPer salvare tutto sul PC locale:')
        print('from google.colab import files')
        print(f"files.download('{exports_archive}')")

print('\nDatasets disponibili:')

for path in sorted((PERSISTENT_ROOT / 'datasets').glob('*')):

    print('-', path)


print('\nRun disponibili:')

for path in sorted((PERSISTENT_ROOT / 'runs').glob('*')):

    print('-', path)

Latest export registry:
run_label: cloud-notebook-smoke-yolov8n-dataset-457d6d2bf0
model_path: exports/cloud-notebook-smoke-yolov8n-dataset-457d6d2bf0.pt
metadata_path: exports/cloud-notebook-smoke-yolov8n-dataset-457d6d2bf0.yaml


Exports disponibili:
- \content\yolo-fire-detector\exports\cloud-notebook-smoke-yolov8n-dataset-457d6d2bf0.pt
- \content\yolo-fire-detector\exports\cloud-notebook-smoke-yolov8n-dataset-457d6d2bf0.yaml
- \content\yolo-fire-detector\exports\latest.yaml

Archivio export pronto: \content\yolo-fire-detector\downloads\yolo-fire-detector-exports.zip

Datasets disponibili:
- \content\yolo-fire-detector\datasets\cloud-notebook-smoke-dataset-457d6d2bf0

Run disponibili:
- \content\yolo-fire-detector\runs\cloud-notebook-smoke-yolov8n-dataset-457d6d2bf0
